# BPE Tokenizer from Scratch

Implementing Byte Pair Encoding (BPE) tokenization algorithm from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re

## 1. Character-Level Tokenization

In [ ]:
text = "the cat sat on the mat. the cat is fat."

# Character-level tokenization
chars = list(text)
char_vocab = sorted(set(chars))
print(f"Text length: {len(chars)} characters")
print(f"Character vocabulary size: {len(char_vocab)}")
print(f"Vocabulary: {char_vocab}")

# Character to index mapping
char_to_idx = {ch: i for i, ch in enumerate(char_vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# Encode text as character indices
encoded = [char_to_idx[ch] for ch in text]
print(f"\nEncoded (first 20): {encoded[:20]}")
print(f"Decoded back: {''.join([idx_to_char[i] for i in encoded[:20]])}")


## 2. BPE Algorithm - Core Implementation

In [ ]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}  # (pair) -> merged token
        self.vocab = {}   # token -> index
        self.inverse_vocab = {}  # index -> token
        self.merge_history = []  # track merge operations
    
    def get_pairs(self, word):
        """Get all adjacent pairs in a word (represented as tuple of tokens)."""
        pairs = Counter()
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += 1
        return pairs
    
    def get_corpus_pairs(self, corpus):
        """Count pair frequencies across the entire corpus."""
        pairs = Counter()
        for word, freq in corpus.items():
            symbols = word
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs
    
    def merge_pair(self, pair, corpus):
        """Merge a pair in all words of the corpus."""
        new_corpus = {}
        bigram = pair
        replacement = pair[0] + pair[1]
        
        for word, freq in corpus.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == bigram[0] and word[i+1] == bigram[1]:
                    new_word.append(replacement)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_corpus[tuple(new_word)] = freq
        return new_corpus
    
    def train(self, text, num_merges=50):
        """Train BPE on the given text."""
        # Tokenize into words, add end-of-word marker
        words = text.split()
        word_freqs = Counter(words)
        
        # Initialize corpus: each word is a tuple of characters + end marker
        corpus = {}
        for word, freq in word_freqs.items():
            corpus[tuple(list(word) + ['</w>'])] = freq
        
        # Build initial character vocabulary
        vocab = set()
        for word in corpus:
            for ch in word:
                vocab.add(ch)
        
        vocab_sizes = [len(vocab)]
        
        # Perform merges
        for i in range(num_merges):
            pairs = self.get_corpus_pairs(corpus)
            if not pairs:
                break
            
            best_pair = pairs.most_common(1)[0][0]
            corpus = self.merge_pair(best_pair, corpus)
            
            merged_token = best_pair[0] + best_pair[1]
            vocab.add(merged_token)
            self.merges[best_pair] = merged_token
            self.merge_history.append((best_pair, pairs.most_common(1)[0][1]))
            vocab_sizes.append(len(vocab))
        
        # Build final vocabulary
        self.vocab = {token: idx for idx, token in enumerate(sorted(vocab))}
        self.inverse_vocab = {idx: token for token, idx in self.vocab.items()}
        self.corpus = corpus
        self.vocab_sizes = vocab_sizes
        
        return corpus


## 3. Training the BPE Tokenizer

In [ ]:
# Training corpus
training_text = """the cat sat on the mat the cat is fat
the dog sat on the log the dog is big
a cat and a dog sat on the mat together
the fat cat sat on the big dog
low lower lowest new newer newest
walking walked walker talks talked talker"""

tokenizer = BPETokenizer()
corpus = tokenizer.train(training_text, num_merges=30)

print("Merge history (first 15 merges):")
print("-" * 50)
for i, (pair, freq) in enumerate(tokenizer.merge_history[:15]):
    print(f"Merge {i+1}: '{pair[0]}' + '{pair[1]}' -> '{pair[0]+pair[1]}' (freq: {freq})")


In [ ]:
print(f"\nFinal vocabulary size: {len(tokenizer.vocab)}")
print(f"Number of merges performed: {len(tokenizer.merge_history)}")
print(f"\nSample vocabulary tokens (sorted by length):")
sorted_vocab = sorted(tokenizer.vocab.keys(), key=len, reverse=True)
for token in sorted_vocab[:20]:
    print(f"  '{token}' (length {len(token)})")


## 4. Encoding and Decoding Text

In [ ]:
def encode_word(word, merges):
    """Encode a single word using learned BPE merges."""
    tokens = list(word) + ['</w>']
    
    while len(tokens) > 1:
        # Find the highest priority merge that can be applied
        best_pair = None
        best_idx = None
        
        for i, (pair, _) in enumerate(merges):
            for j in range(len(tokens) - 1):
                if tokens[j] == pair[0] and tokens[j+1] == pair[1]:
                    if best_idx is None or i < best_idx:
                        best_pair = pair
                        best_idx = i
                    break
        
        if best_pair is None:
            break
        
        # Apply the merge
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == best_pair[0] and tokens[i+1] == best_pair[1]:
                new_tokens.append(best_pair[0] + best_pair[1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    return tokens

def encode_text(text, merges):
    """Encode full text using BPE."""
    words = text.split()
    all_tokens = []
    for word in words:
        tokens = encode_word(word, merges)
        all_tokens.extend(tokens)
    return all_tokens

def decode_tokens(tokens):
    """Decode BPE tokens back to text."""
    text = ''.join(tokens)
    text = text.replace('</w>', ' ')
    return text.strip()

# Test encoding/decoding
merge_list = [(pair, freq) for pair, freq in zip(tokenizer.merges.keys(), 
              [f for _, f in tokenizer.merge_history])]

test_text = "the cat sat on the mat"
encoded = encode_text(test_text, tokenizer.merge_history)
decoded = decode_tokens(encoded)

print(f"Original: '{test_text}'")
print(f"Tokens:   {encoded}")
print(f"Decoded:  '{decoded}'")
print(f"Num tokens: {len(encoded)} (vs {len(test_text)} characters)")


## 5. Visualize Vocabulary Growth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vocabulary size growth
axes[0].plot(range(len(tokenizer.vocab_sizes)), tokenizer.vocab_sizes, 'b-o', markersize=4)
axes[0].set_xlabel('Number of Merges')
axes[0].set_ylabel('Vocabulary Size')
axes[0].set_title('Vocabulary Growth During BPE Training')
axes[0].grid(True, alpha=0.3)

# Merge frequencies
merge_freqs = [f for _, f in tokenizer.merge_history]
axes[1].bar(range(len(merge_freqs)), merge_freqs, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Merge Operation Index')
axes[1].set_ylabel('Pair Frequency')
axes[1].set_title('Frequency of Merged Pairs (Decreasing)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## 6. Compare Tokenization of Different Texts

In [ ]:
test_cases = [
    ("English", "the cat walked on the lowest mat"),
    ("Code-like", "def function_name(x, y): return x + y"),
    ("Numbers", "the price is 12345 dollars and 67 cents"),
    ("Unknown words", "supercalifragilistic xyzzy abracadabra"),
]

print("Tokenization Comparison")
print("=" * 70)
for category, text in test_cases:
    tokens = encode_text(text, tokenizer.merge_history)
    compression = len(text) / max(len(tokens), 1)
    print(f"\n[{category}] '{text}'")
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print(f"  Compression ratio: {compression:.2f}x (chars/token)")


## 7. Handling Unknown Words with Subword Tokenization

In [ ]:
# Demonstrate how BPE handles out-of-vocabulary words
unknown_words = ["walking", "unwalked", "rewalking", "catdog", "lowest", "unhappiness"]

print("Subword Tokenization of Unknown/Rare Words")
print("=" * 60)
for word in unknown_words:
    tokens = encode_word(word, tokenizer.merge_history)
    print(f"  '{word}' -> {tokens}")

print("\n\nKey insight: BPE never produces <UNK> tokens!")
print("It falls back to character-level for truly unknown substrings.")
print("Common subwords (like 'ing', 'ed', 'er') get their own tokens.")


## 8. Token Frequency Analysis (Zipf's Law)

In [ ]:
# Tokenize a larger corpus and analyze token frequencies
large_text = (training_text + " ") * 10 + "the quick brown fox jumps over the lazy dog " * 5
all_tokens = encode_text(large_text, tokenizer.merge_history)

token_counts = Counter(all_tokens)
sorted_counts = token_counts.most_common()

ranks = np.arange(1, len(sorted_counts) + 1)
frequencies = np.array([count for _, count in sorted_counts])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regular scale
axes[0].bar(range(min(25, len(sorted_counts))), frequencies[:25], color='coral', alpha=0.7)
axes[0].set_xlabel('Token Rank')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Token Frequency Distribution (Top 25)')
labels = [t for t, _ in sorted_counts[:25]]
axes[0].set_xticks(range(min(25, len(sorted_counts))))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[0].grid(True, alpha=0.3, axis='y')

# Log-log scale (Zipf's law)
axes[1].loglog(ranks, frequencies, 'bo-', markersize=3, alpha=0.7)
# Zipf reference line
zipf_ref = frequencies[0] / ranks
axes[1].loglog(ranks, zipf_ref, 'r--', alpha=0.5, label='Zipf (1/rank)')
axes[1].set_xlabel('Rank (log scale)')
axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_title("Zipf's Law in Token Frequencies")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal tokens: {len(all_tokens)}")
print(f"Unique tokens: {len(token_counts)}")
print(f"Top 10 tokens cover {sum(frequencies[:10])/sum(frequencies)*100:.1f}% of all occurrences")


## 9. Effect of Vocabulary Size on Compression

In [ ]:
# Train tokenizers with different numbers of merges
merge_counts = [0, 5, 10, 20, 30, 50, 75, 100]
test_sentence = "the cat walked on the lowest mat and talked to the dog"

results = []
for n_merges in merge_counts:
    tok = BPETokenizer()
    tok.train(training_text, num_merges=n_merges)
    tokens = encode_text(test_sentence, tok.merge_history)
    results.append({
        'merges': n_merges,
        'vocab_size': len(tok.vocab),
        'num_tokens': len(tokens),
        'avg_token_len': np.mean([len(t.replace('</w>', '')) for t in tokens])
    })

fig, ax1 = plt.subplots(figsize=(10, 5))

merges_arr = [r['merges'] for r in results]
tokens_arr = [r['num_tokens'] for r in results]
vocab_arr = [r['vocab_size'] for r in results]

ax1.plot(merges_arr, tokens_arr, 'b-o', label='Tokens needed')
ax1.set_xlabel('Number of BPE Merges')
ax1.set_ylabel('Number of Tokens (for test sentence)', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(merges_arr, vocab_arr, 'r-s', label='Vocabulary size')
ax2.set_ylabel('Vocabulary Size', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('Trade-off: More Merges = Fewer Tokens but Larger Vocabulary')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.85))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Summary

Key takeaways:
- **BPE** is a data-driven tokenization algorithm that learns subword units
- It starts with characters and iteratively merges the most frequent pairs
- Handles OOV words gracefully by falling back to smaller subwords
- Token frequency follows Zipf's law
- Vocabulary size is a tunable hyperparameter (trade-off between sequence length and vocab size)
- Modern LLMs (GPT, LLaMA) use variants of BPE with ~32K-100K vocabulary sizes

In [ ]:
# Final comparison table
import pandas as pd

comparison_data = {
    'Method': ['Character-level', 'BPE (30 merges)', 'BPE (100 merges)', 'Word-level'],
    'Vocab Size': [len(char_vocab), len(tokenizer.vocab), vocab_arr[-1], len(set(training_text.split()))],
    'Handles OOV': ['Always', 'Always', 'Always', 'No (<UNK>)'],
    'Sequence Length': ['Very long', 'Medium', 'Shorter', 'Shortest'],
    'Semantic Units': ['None', 'Subwords', 'Larger subwords', 'Full words'],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))
